# SDM invullen

### Imports

In [41]:
import sqlite3
import pandas as pd
import os

### Verbinding maken met SDM

In [ ]:
SDM = sqlite3.connect("../../Database/SDM2.db")
cursor = SDM.cursor()

#alle databases verbinding 
dbs = {
    "Accessoireverkoop": sqlite3.connect("../../Database/BikeToDrive_1_Accessoireverkoop.db"),
    "Fietsverkoop": sqlite3.connect("../../Database/BikeToDrive_2_Fietsverkoop.db"),
    "Onderhoud": sqlite3.connect("../../Database/BikeToDrive_3_Onderhoud.db"),
    "Accessoire_Inkoop": sqlite3.connect("../../Database/BikeToDrive_4_Accessoire_Inkoop.db"),
    "Fiets_Inkoop": sqlite3.connect("../../Database/BikeToDrive_5_Fiets_Inkoop.db"),
}

### Load Data


In [ ]:

#Alle tabellen selecteren 
def get_tables(conn):
    #read
    return pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
        #lijst met alle tabel namen 
    )['name'].tolist()


#laad tabel 
def load_db(conn, name):
    data = {}

    # For alle tabel in dit db 
    for t in get_tables(conn):

        #read => into dataframe
        df = pd.read_sql_query(f"SELECT * FROM {t}", conn)

        #unique key 
        key = f"{name}_{t}"

        # Store DataFrame in dictionary
        data[key] = df

        #controleren of het ingeladen is
        print(f"Loaded: {key}")
    return data

#combinneer alle data => 1 dictionary
all_data = {}

for name, conn in dbs.items():
    all_data.update(load_db(conn, name))


Loaded: Accessoireverkoop_Accessoire_Verkoop
Loaded: Accessoireverkoop_Monteur
Loaded: Accessoireverkoop_Leverancier
Loaded: Accessoireverkoop_Accessoire
Loaded: Accessoireverkoop_Filiaal
Loaded: Accessoireverkoop_Klant
Loaded: Fietsverkoop_Fiets_Verkoop
Loaded: Fietsverkoop_Fiets
Loaded: Fietsverkoop_Monteur
Loaded: Fietsverkoop_Fabrikant
Loaded: Fietsverkoop_Filiaal
Loaded: Fietsverkoop_Klant
Loaded: Onderhoud_Onderhoud
Loaded: Onderhoud_Fiets
Loaded: Onderhoud_Monteur
Loaded: Onderhoud_Fabrikant
Loaded: Onderhoud_Filiaal
Loaded: Accessoire_Inkoop_Accessoire_Inkoop
Loaded: Accessoire_Inkoop_Accessoire
Loaded: Accessoire_Inkoop_Leverancier
Loaded: Fiets_Inkoop_Fiets_Inkoop
Loaded: Fiets_Inkoop_Fiets
Loaded: Fiets_Inkoop_Fabrikant


Dit zorgt er voor dat we later makelijker dingen kunnen doen zoals:
1. automatisch tabellen maken
2. automatisch tabellen leegmaken
3. automatisch data laden

### Reset_knop
a.	Alle tabellen uit het SDM leegmaken. Dit is dus een soort “reset-knop” voor de invulling van je SDM-tabellen.

In [ ]:
#reset functie 
def reset_sdm(conn):
    #SDM => alle tabel name
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )['name'].tolist()

    #reverse => foreign key issue oplosen 
    for t in reversed(tables):
        try:
            conn.execute(f"DELETE FROM {t}")
            print(f"Cleared: {t}")
        except:
            pass

    conn.commit()

#run
reset_sdm(SDM)


Cleared: Fiets_Inkoop
Cleared: Fiets_Inkoop_Fiets
Cleared: Fiets_Inkoop_Fabrikant
Cleared: Accessoire_Inkoop
Cleared: Accessoire_Inkoop_Accessoire
Cleared: Accessoire_Inkoop_Leverancier
Cleared: Onderhoud
Cleared: Onderhoud_Monteur
Cleared: Onderhoud_Filiaal
Cleared: Onderhoud_Fiets
Cleared: Onderhoud_Fabrikant
Cleared: Fietsverkoop_Fiets_Verkoop
Cleared: Fietsverkoop_Fiets
Cleared: Fietsverkoop_Monteur
Cleared: Fietsverkoop_Klant
Cleared: Fietsverkoop_Filiaal
Cleared: Fietsverkoop_Fabrikant
Cleared: Accessoireverkoop_Accessoire_Verkoop
Cleared: Accessoireverkoop_Accessoire
Cleared: Accessoireverkoop_Klant
Cleared: Accessoireverkoop_Monteur
Cleared: Accessoireverkoop_Filiaal
Cleared: Accessoireverkoop_Leverancier


### Mapping 

In [ ]:
#Mapping => tussen SDM tables en source entities 
SDM_MAPPING = {
    #entity
    "Leverancier": [
        "Accessoireverkoop_Leverancier", #tabel
        "Accessoire_Inkoop_Leverancier"
    ],

    "Filiaal": [
        "Accessoireverkoop_Filiaal",
        "Fietsverkoop_Filiaal",
        "Onderhoud_Filiaal"
    ],

    "Fabrikant": [
        "Fietsverkoop_Fabrikant",
        "Onderhoud_Fabrikant",
        "Fiets_Inkoop_Fabrikant"
    ],

    "Klant": [
        "Accessoireverkoop_Klant",
        "Fietsverkoop_Klant"
    ],

    "Monteur": [
        "Accessoireverkoop_Monteur",
        "Fietsverkoop_Monteur",
        "Onderhoud_Monteur"
    ],

    "Accessoire": [
        "Accessoireverkoop_Accessoire",
        "Accessoire_Inkoop_Accessoire"
    ],

    "Fiets": [
        "Fietsverkoop_Fiets",
        "Onderhoud_Fiets",
        "Fiets_Inkoop_Fiets"
    ]
}


insert to sdm

In [ ]:
#Insert Functie => DataFrame into table
def insert(df, table, conn):
    # Append data to table
    df.to_sql(table, conn, if_exists="append", index=False)
    print(f"Inserted → {table}")


Load dimensions


In [ ]:
# Laad Function => laad dimension tables
def load_dimensions(mapping, data, conn):
     #Loop => elke entity
    for entity, tables in mapping.items():
        # Loop door SDM tables => waar entiteit moet
        for table in tables:

            source_key = None

             # vind matching source table in all_data
            for k in data.keys():
                if k.endswith(f"_{entity}"):
                    source_key = k

            # If found => insert into SDM
            if source_key:
                insert(data[source_key], table, conn)
            else:
                print(f"Missing: {entity}")

# run 
load_dimensions(SDM_MAPPING, all_data, SDM)


Inserted → Accessoireverkoop_Leverancier
Inserted → Accessoire_Inkoop_Leverancier
Inserted → Accessoireverkoop_Filiaal
Inserted → Fietsverkoop_Filiaal
Inserted → Onderhoud_Filiaal
Inserted → Fietsverkoop_Fabrikant
Inserted → Onderhoud_Fabrikant
Inserted → Fiets_Inkoop_Fabrikant
Inserted → Accessoireverkoop_Klant
Inserted → Fietsverkoop_Klant
Inserted → Accessoireverkoop_Monteur
Inserted → Fietsverkoop_Monteur
Inserted → Onderhoud_Monteur
Inserted → Accessoireverkoop_Accessoire
Inserted → Accessoire_Inkoop_Accessoire
Inserted → Fietsverkoop_Fiets
Inserted → Onderhoud_Fiets
Inserted → Fiets_Inkoop_Fiets


Load facts

In [ ]:
# Mapping => feiten
fact_tables = {
    "Onderhoud": "Onderhoud_Onderhoud",
    "Accessoire_Inkoop": "Accessoire_Inkoop_Accessoire_Inkoop",
    "Fiets_Inkoop": "Fiets_Inkoop_Fiets_Inkoop",
    "Accessoireverkoop_Accessoire_Verkoop": "Accessoireverkoop_Accessoire_Verkoop",
    "Fietsverkoop_Fiets_Verkoop": "Fietsverkoop_Fiets_Verkoop"
}


for target, source in fact_tables.items():
     # controleren of tabel bestaat
    if source in all_data:
        # Insert into SDM
        all_data[source].to_sql(target, SDM, if_exists="append", index=False)
        print(f"FACT LOADED → {target}")


FACT LOADED → Onderhoud
FACT LOADED → Accessoire_Inkoop
FACT LOADED → Fiets_Inkoop
FACT LOADED → Accessoireverkoop_Accessoire_Verkoop
FACT LOADED → Fietsverkoop_Fiets_Verkoop


In [49]:
SDM.commit()


controleren of er daadwerkelijk rijen bestaan => er is een sdm

In [ ]:
#row count per tabel controleren
def check_counts(conn):
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )['name'].tolist()

    for t in tables:
        count = pd.read_sql_query(f"SELECT COUNT(*) as c FROM {t}", conn)['c'][0]
        print(f"{t}: {count}")


check_counts(SDM)


Accessoireverkoop_Leverancier: 5
Accessoireverkoop_Filiaal: 5
Accessoireverkoop_Monteur: 15
Accessoireverkoop_Klant: 25
Accessoireverkoop_Accessoire: 13
Accessoireverkoop_Accessoire_Verkoop: 100
Fietsverkoop_Fabrikant: 10
Fietsverkoop_Filiaal: 5
Fietsverkoop_Klant: 25
Fietsverkoop_Monteur: 15
Fietsverkoop_Fiets: 75
Fietsverkoop_Fiets_Verkoop: 150
Onderhoud_Fabrikant: 10
Onderhoud_Fiets: 75
Onderhoud_Filiaal: 5
Onderhoud_Monteur: 15
Onderhoud: 50
Accessoire_Inkoop_Leverancier: 5
Accessoire_Inkoop_Accessoire: 13
Accessoire_Inkoop: 50
Fiets_Inkoop_Fabrikant: 10
Fiets_Inkoop_Fiets: 75
Fiets_Inkoop: 100


In [ ]:
#SDM tabellen namen zien

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    SDM
)

print(tables)


                                    name
0          Accessoireverkoop_Leverancier
1              Accessoireverkoop_Filiaal
2              Accessoireverkoop_Monteur
3                Accessoireverkoop_Klant
4           Accessoireverkoop_Accessoire
5   Accessoireverkoop_Accessoire_Verkoop
6                 Fietsverkoop_Fabrikant
7                   Fietsverkoop_Filiaal
8                     Fietsverkoop_Klant
9                   Fietsverkoop_Monteur
10                    Fietsverkoop_Fiets
11            Fietsverkoop_Fiets_Verkoop
12                   Onderhoud_Fabrikant
13                       Onderhoud_Fiets
14                     Onderhoud_Filiaal
15                     Onderhoud_Monteur
16                             Onderhoud
17         Accessoire_Inkoop_Leverancier
18          Accessoire_Inkoop_Accessoire
19                     Accessoire_Inkoop
20                Fiets_Inkoop_Fabrikant
21                    Fiets_Inkoop_Fiets
22                          Fiets_Inkoop
